# Batch Inference — Instacart Reorder Prediction

Loads the Champion model from MLflow Model Registry, scores all user-product pairs in `gold.serving_features`, and writes predictions back to Iceberg.

**Input:** `AwsDataCatalog.gold.serving_features` (Iceberg)
**Output:**
- `gold.predictions_audit` — all scores, appended, partitioned by `prediction_date`
- `gold.predictions_top_n` — top 10 recommendations per user, overwritten each run

## 1. Setup

In [ ]:
import awswrangler as wr
import mlflow
import mlflow.xgboost
import pandas as pd
from datetime import date

BUCKET_NAME = "instacart-aws-data-ml-eng-project"
ATHENA_OUTPUT = f"s3://{BUCKET_NAME}/athena-results/"
DATABASE = "gold"
MODEL_NAME = "instacart-reorder-model"
MLFLOW_TRACKING_URI = "arn:aws:sagemaker:ap-southeast-2:606476261726:mlflow-tracking-server/instacart-mlflow"
TOP_N = 10

FEATURE_COLUMNS = [
    "user_product_buy_count",
    "user_product_last_order_in_window",
    "user_product_avg_add_to_cart",
    "user_product_orders_since_last_buy",
    "user_window_orders",
    "user_avg_days_between",
    "user_total_orders",
    "user_product_window_reorder_ratio",
    "item_total_sales",
    "item_reorder_rate",
]

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print("Setup complete.")

## 2. Load Champion Model

In [ ]:
client = mlflow.tracking.MlflowClient()
champion = client.get_model_version_by_alias(MODEL_NAME, "Champion")
model_version = int(champion.version)

model_uri = f"models:/{MODEL_NAME}@Champion"
loaded_model = mlflow.xgboost.load_model(model_uri)
print(f"Loaded model '{MODEL_NAME}' version {model_version} (run_id={champion.run_id})")

## 3. Score Serving Features

In [ ]:
serving_df = wr.athena.read_sql_query(
    f"SELECT user_id, product_id, {', '.join(FEATURE_COLUMNS)} FROM gold.serving_features",
    database=DATABASE,
    s3_output=ATHENA_OUTPUT,
)
print(f"Scoring {len(serving_df):,} user-product pairs")

serving_df["reorder_probability"] = loaded_model.predict_proba(serving_df[FEATURE_COLUMNS])[:, 1]
serving_df["model_version"] = model_version
serving_df["prediction_date"] = str(date.today())

predictions_df = serving_df[["user_id", "product_id", "reorder_probability", "model_version", "prediction_date"]]
print(f"Score range: {predictions_df['reorder_probability'].min():.4f} – {predictions_df['reorder_probability'].max():.4f}")

## 4. Write predictions_audit

In [ ]:
wr.athena.to_iceberg(
    df=predictions_df,
    database=DATABASE,
    table="predictions_audit",
    temp_path=f"{ATHENA_OUTPUT}temp/",
    table_location=f"s3://{BUCKET_NAME}/gold/predictions_audit/",
    keep_files=False,
    partition_cols=["prediction_date"],
    mode="append",
)
print(f"✅ predictions_audit: {len(predictions_df):,} rows written (partition={date.today()})")

## 5. Write predictions_top_n

In [ ]:
top_n_df = (
    predictions_df
    .sort_values("reorder_probability", ascending=False)
    .groupby("user_id")
    .head(TOP_N)
    .reset_index(drop=True)
)

wr.athena.to_iceberg(
    df=top_n_df,
    database=DATABASE,
    table="predictions_top_n",
    temp_path=f"{ATHENA_OUTPUT}temp/",
    table_location=f"s3://{BUCKET_NAME}/gold/predictions_top_n/",
    keep_files=False,
    mode="overwrite",
)
print(f"✅ predictions_top_n: {len(top_n_df):,} rows written (top {TOP_N} per user)")

## 6. Verify

In [ ]:
wr.athena.read_sql_query(
    "SELECT COUNT(*) AS total FROM gold.predictions_top_n",
    database=DATABASE, s3_output=ATHENA_OUTPUT,
)

In [ ]:
wr.athena.read_sql_query(
    "SELECT user_id, COUNT(*) AS cnt FROM gold.predictions_top_n GROUP BY user_id ORDER BY user_id LIMIT 5",
    database=DATABASE, s3_output=ATHENA_OUTPUT,
)

In [ ]:
wr.athena.read_sql_query(
    "SELECT prediction_date, COUNT(*) AS cnt FROM gold.predictions_audit GROUP BY prediction_date ORDER BY prediction_date DESC",
    database=DATABASE, s3_output=ATHENA_OUTPUT,
)